### RAG PIPELINE- Data Ingestion to Vector DB

In [1]:
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

c:\Users\nisha\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from collections import Counter
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


def load_pdfs(directory_path):
    """
    Load all PDF files and print summary.
    """
    
    # Count PDF files first
    pdf_files = []
    for root, _, files in os.walk(directory_path):
        for file in files:
            if file.endswith(".pdf"):
                pdf_files.append(os.path.join(root, file))
    
    print(f"\n📄 Found {len(pdf_files)} PDF files to process\n")
    
    # Load PDFs
    loader = DirectoryLoader(
        directory_path,
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress=False
    )
    
    documents = loader.load()
    
    # Count pages per PDF
    counts = Counter([doc.metadata["source"] for doc in documents])
    
    print("\n📑 PDF Details:")
    for i, (pdf, count) in enumerate(counts.items(), start=1):
        file_name = os.path.basename(pdf)
        print(f"{i}. {file_name} → {count} pages")
    
    # Total documents
    print(f"\n📊 Total documents (pages): {len(documents)}\n")
    
    return documents


# 🔥 Call the function
documents = load_pdfs("../data")


📄 Found 3 PDF files to process


📑 PDF Details:
1. Attention.pdf → 15 pages
2. Embeddings.pdf → 10 pages
3. Retrieval.pdf → 84 pages

📊 Total documents (pages): 109



In [3]:
documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': '..\\data\\pdf\\Attention.pdf', 'file_path': '..\\data\\pdf\\Attention.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle

In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Split documents into chunks and preserve metadata.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    all_chunks = []

    for doc in documents:
        chunks = text_splitter.split_text(doc.page_content)

        for chunk in chunks:
            chunk_doc = Document(
                page_content=chunk,
                metadata=doc.metadata
            )
            all_chunks.append(chunk_doc)

    print(f"\n✂️ Split into {len(all_chunks)} chunks (size: {chunk_size}, overlap: {chunk_overlap})\n")

    return all_chunks

In [5]:
chunks = split_documents(documents)


✂️ Split into 312 chunks (size: 1000, overlap: 200)



### Embedding and vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
from __future__ import annotations

from typing import List, Optional
from sentence_transformers import SentenceTransformer
import numpy as np


class EmbeddingManager:
    """Handle text embeddings using a Sentence Transformers model."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2", device: Optional[str] = None):
        self.model_name = model_name
        self.device = device
        self.model: Optional[SentenceTransformer] = None
        self._load_model()

    def _load_model(self) -> None:
        try:
            print(f"🔄 Loading embedding model: {self.model_name}...")
            if self.device:
                self.model = SentenceTransformer(self.model_name, device=self.device)
            else:
                self.model = SentenceTransformer(self.model_name)
            print("✅ Model loaded successfully!")
            print(f"📏 Embedding dimension: {self.get_sentence_embedding_dimension()}")
        except Exception as e:
            raise RuntimeError(
                f"Failed to load embedding model '{self.model_name}'."
            ) from e

    def get_sentence_embedding_dimension(self) -> int:
        if self.model is None:
            raise RuntimeError("Model is not loaded.")
        return self.model.get_sentence_embedding_dimension()

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not texts:
            raise ValueError("Input texts list cannot be empty.")

        if self.model is None:
            raise RuntimeError("Model is not loaded.")

        cleaned_texts = [t.strip() for t in texts if t and t.strip()]
        if not cleaned_texts:
            raise ValueError("No valid texts found.")

        print(f"🔄 Generating embeddings for {len(cleaned_texts)} texts...")

        embeddings = self.model.encode(
            cleaned_texts,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        print("✅ Embeddings generated!")
        print("Embeddings shape:", embeddings.shape)

        expected_dim = self.get_sentence_embedding_dimension()
        actual_dim = embeddings.shape[1]

        if actual_dim != expected_dim:
            raise ValueError(
                f"Embedding dimension mismatch: expected {expected_dim}, got {actual_dim}"
            )

        print(f"✅ Verified embedding dimension: {actual_dim}")

        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()

# Test it
texts = ["RAG is powerful", "Embeddings convert text into vectors"]
embeddings = embedding_manager.generate_embeddings(texts)

print("Final shape:", embeddings.shape)

🔄 Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4426.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully!
📏 Embedding dimension: 384
🔄 Generating embeddings for 2 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.51s/it]

✅ Embeddings generated!
Embeddings shape: (2, 384)
✅ Verified embedding dimension: 384
Final shape: (2, 384)


### VectorStore

In [8]:
import uuid
from typing import List, Any

import chromadb
from chromadb.config import Settings
import numpy as np


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store."""

    def __init__(
        self,
        collection_name: str = "pdf_collection",
        persist_directory: str = "vector_store",
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_vector_store()

    def _initialize_vector_store(self) -> None:
        try:
            print(f"🔄 Initializing ChromaDB vector store at '{self.persist_directory}'...")

            # Modern persistent Chroma client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory,
                settings=Settings(anonymized_telemetry=False)
            )

            # Get existing collection names safely
            existing_collections = [c.name for c in self.client.list_collections()]

            if self.collection_name in existing_collections:
                print(f"⚠️ Collection '{self.collection_name}' already exists. It will be overwritten.")
                self.client.delete_collection(self.collection_name)

            self.collection = self.client.create_collection(name=self.collection_name)

            print("✅ Vector store initialized successfully!")

        except Exception as e:
            raise RuntimeError(f"Failed to initialize vector store: {e}") from e

    def add_documents(self, documents: List[Any], embeddings: np.ndarray) -> None:
        """
        Add documents and their embeddings to the vector store.
        """
        if not documents:
            raise ValueError("Documents list cannot be empty.")

        if embeddings is None or len(embeddings) == 0:
            raise ValueError("Embeddings cannot be empty.")

        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings must have same length.")

        if self.collection is None:
            raise RuntimeError("Collection is not initialized.")

        print(f"🔄 Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [str(uuid.uuid4()) for _ in range(len(documents))]
        embeddings_list = embeddings.tolist()

        # Store in ChromaDB
        self.collection.add(
            ids=ids,
            documents=texts,
            metadatas=metadatas,
            embeddings=embeddings_list,
        )

        print(f"✅ Added {len(documents)} documents to ChromaDB successfully!")
    
    def search(self, query: str, embedding_manager, k: int = 3):
        """
        Search top-k similar documents.
        """
        print(f"\n🔍 Searching for: '{query}'")

        # Step 1: embed query
        query_embedding = embedding_manager.generate_embeddings([query])

        # Step 2: search in Chroma
        results = self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=k
        )

        # Step 3: display results
        print(f"\n📌 Top {k} results:\n")

        for i in range(len(results["documents"][0])):
            print(f"--- Result {i+1} ---")
            print("Text:", results["documents"][0][i][:300])
            print("Metadata:", results["metadatas"][0][i])
            print()

        return results
        
vector_store = VectorStore()
vector_store

🔄 Initializing ChromaDB vector store at 'vector_store'...
⚠️ Collection 'pdf_collection' already exists. It will be overwritten.
✅ Vector store initialized successfully!


In [9]:
### Convert the text to embeddings

documents = load_pdfs("../data")

chunks = split_documents(documents)

texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embeddings)


📄 Found 3 PDF files to process


📑 PDF Details:
1. Attention.pdf → 15 pages
2. Embeddings.pdf → 10 pages
3. Retrieval.pdf → 84 pages

📊 Total documents (pages): 109


✂️ Split into 312 chunks (size: 1000, overlap: 200)

🔄 Generating embeddings for 312 texts...


Batches: 100%|██████████| 10/10 [00:39<00:00,  3.98s/it]


✅ Embeddings generated!
Embeddings shape: (312, 384)
✅ Verified embedding dimension: 384
🔄 Adding 312 documents to vector store...
✅ Added 312 documents to ChromaDB successfully!


In [10]:
vector_store.search(
    "What is attention in transformers?",
    embedding_manager,
    k=3
)


🔍 Searching for: 'What is attention in transformers?'
🔄 Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]

✅ Embeddings generated!
Embeddings shape: (1, 384)
✅ Verified embedding dimension: 384

📌 Top 3 results:

--- Result 1 ---
Text: Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the d
Metadata: {'page': 4, 'producer': 'pdfTeX-1.40.25', 'keywords': '', 'creator': 'LaTeX with hyperref', 'total_pages': 15, 'subject': '', 'modDate': 'D:20240410211143Z', 'file_path': '..\\data\\pdf\\Attention.pdf', 'source': '..\\data\\pdf\\Attention.pdf', 'creationDate': 'D:20240410211143Z', 'title': '', 'format': 'PDF 1.5', 'author': '', 'trapped': '', 'moddate': '2024-04-10T21:11:43+00:00', 'creationdate': '2024-04-10T21:11:43+00:00'}

--- Result 2 ---
Text: 3.2
Attention
An attention function can be described as mapping a query and a set of key-value pairs to an

{'ids': [['c565ae9b-d9c1-4d9b-a0f8-a51133ceb85b',
   'c944aca8-8a23-4b53-a00b-bb07f85b6a0b',
   'bd3a0de5-785d-43cd-8efe-18eb1da4a30c']],
 'embeddings': None,
 'documents': [['Applications of Attention in our Model\nThe Transformer uses multi-head attention in three different ways:\n• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,\nand the memory keys and values come from the output of the encoder. This allows every\nposition in the decoder to attend over all positions in the input sequence. This mimics the\ntypical encoder-decoder attention mechanisms in sequence-to-sequence models such as\n[38, 2, 9].\n• The encoder contains self-attention layers. In a self-attention layer all of the keys, values\nand queries come from the same place, in this case, the output of the previous layer in the\nencoder. Each position in the encoder can attend to all positions in the previous layer of the\nencoder.\n• Similarly, self-attention layers in the decoder 

## Retriever Pipeline From VectorStore

In [11]:
from typing import List, Dict, Any, Optional


class RagRetriever:
    """Handle query-based retrieval from the vector store."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store used for similarity search.
            embedding_manager: Embedding manager used to encode queries.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        k: int = 3,
        score_threshold: Optional[float] = None
    ) -> List[Dict[str, Any]]:
        """
        Retrieve top-k relevant chunks for a query.

        Args:
            query: Query string to search for.
            k: Number of results to retrieve.
            score_threshold: Maximum allowed distance for a result.
                Lower distance means more relevant in Chroma.
                Use None to return all retrieved results.

        Returns:
            A list of dictionaries containing retrieved text, metadata,
            and distance score when available.
        """
        if not query or not query.strip():
            raise ValueError("Query cannot be empty.")

        if k <= 0:
            raise ValueError("k must be greater than 0.")

        # Perform vector search
        results = self.vector_store.search(
            query=query,
            embedding_manager=self.embedding_manager,
            k=k
        )

        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])[0] if "distances" in results else [None] * len(documents)

        retrieved_results = []

        for doc, metadata, distance in zip(documents, metadatas, distances):
            # Apply distance-based filtering if threshold is provided
            if score_threshold is not None and distance is not None:
                if distance > score_threshold:
                    continue

            retrieved_results.append(
                {
                    "text": doc,
                    "metadata": metadata,
                    "score": distance
                }
            )

        return retrieved_results

In [12]:
retriever = RagRetriever(vector_store, embedding_manager)

results = retriever.retrieve(
    query="What is attention in transformers?",
    k=3
)

for i, item in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", item["metadata"].get("source"))
    print("Page:", item["metadata"].get("page"))
    print("Score:", item.get("score"))
    print("Text:", item["text"][:300])


🔍 Searching for: 'What is attention in transformers?'
🔄 Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.17it/s]

✅ Embeddings generated!
Embeddings shape: (1, 384)
✅ Verified embedding dimension: 384

📌 Top 3 results:

--- Result 1 ---
Text: Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the d
Metadata: {'creationDate': 'D:20240410211143Z', 'modDate': 'D:20240410211143Z', 'trapped': '', 'total_pages': 15, 'format': 'PDF 1.5', 'moddate': '2024-04-10T21:11:43+00:00', 'author': '', 'subject': '', 'file_path': '..\\data\\pdf\\Attention.pdf', 'creationdate': '2024-04-10T21:11:43+00:00', 'creator': 'LaTeX with hyperref', 'page': 4, 'keywords': '', 'title': '', 'producer': 'pdfTeX-1.40.25', 'source': '..\\data\\pdf\\Attention.pdf'}

--- Result 2 ---
Text: 3.2
Attention
An attention function can be described as mapping a query and a set of key-value pairs to an

### Integration Vectordb Context pipeline with LLM output

In [ ]:
from typing import List, Dict, Any, Optional
from openai import OpenAI


class RAGPipeline:
    """Integrates retriever, context building, and LLM-based answer generation."""

    def __init__(
        self,
        retriever: RagRetriever,
        model_name: str = "gpt-4o-mini",
        max_context_chars: int = 4000
    ):
        """
        Initialize the RAG pipeline.

        Args:
            retriever: RagRetriever instance for retrieving relevant chunks.
            model_name: OpenAI model name for answer generation.
            max_context_chars: Maximum number of characters to keep in context.
        """
        self.retriever = retriever
        self.model_name = model_name
        self.max_context_chars = max_context_chars
        self.client = OpenAI(api_key="")

    def build_context(self, retrieved_docs: List[Dict[str, Any]]) -> str:
        """
        Build a single context string from retrieved documents.

        Args:
            retrieved_docs: List of retrieved document dictionaries.

        Returns:
            A formatted context string.
        """
        context_parts = []
        total_chars = 0

        for i, doc in enumerate(retrieved_docs, start=1):
            text = doc.get("text", "")
            metadata = doc.get("metadata", {})

            source = metadata.get("source", "Unknown")
            page = metadata.get("page", "N/A")
            chunk_id = metadata.get("chunk_id", "N/A")

            chunk_text = (
                f"[Document {i}]\n"
                f"Source: {source}\n"
                f"Page: {page}\n"
                f"Chunk ID: {chunk_id}\n"
                f"Content:\n{text}\n"
            )

            if total_chars + len(chunk_text) > self.max_context_chars:
                break

            context_parts.append(chunk_text)
            total_chars += len(chunk_text)

        return "\n\n".join(context_parts)

    def generate_answer(self, query: str, context: str) -> str:
        """
        Generate answer using the LLM based on retrieved context.

        Args:
            query: User question.
            context: Retrieved context.

        Returns:
            Generated answer text.
        """
        prompt = f"""
You are a helpful RAG assistant.

Answer the user's question using only the provided context.
If the answer is not in the context, say:
"I could not find the answer in the provided documents."

Context:
{context}

Question:
{query}
"""

        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": "You answer questions only from provided context."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )

        return response.choices[0].message.content

    def run(self, query: str, k: int = 3, score_threshold: Optional[float] = None) -> Dict[str, Any]:
        """
        Run the full RAG pipeline:
        retrieve -> build context -> generate answer

        Args:
            query: User question.
            k: Number of top chunks to retrieve.
            score_threshold: Optional retrieval filtering threshold.

        Returns:
            Dictionary containing query, retrieved docs, context, and final answer.
        """
        print(f"\n🔍 Query: {query}")

        # Step 1: Retrieve relevant documents
        retrieved_docs = self.retriever.retrieve(
            query=query,
            k=k,
            score_threshold=score_threshold
        )

        print(f"✅ Retrieved {len(retrieved_docs)} relevant chunks")

        # Step 2: Build context
        context = self.build_context(retrieved_docs)

        print("✅ Context built successfully")

        # Step 3: Generate final answer
        answer = self.generate_answer(query, context)

        print("✅ Answer generated successfully")

        return {
            "query": query,
            "retrieved_docs": retrieved_docs,
            "context": context,
            "answer": answer
        }

In [17]:
# Initialize retriever
retriever = RagRetriever(vector_store, embedding_manager)

# Initialize RAG pipeline
rag_pipeline = RAGPipeline(retriever=retriever)

# 🔍 Test query
query = "What are precision and recall in information retrieval systems, and how do they differ?"

# Run full pipeline
result = rag_pipeline.run(query=query, k=3)

# 🎯 Print final answer
print("\n🧠 Final Answer:\n")
print(result["answer"])


🔍 Query: What are precision and recall in information retrieval systems, and how do they differ?

🔍 Searching for: 'What are precision and recall in information retrieval systems, and how do they differ?'
🔄 Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 68.72it/s]

✅ Embeddings generated!
Embeddings shape: (1, 384)
✅ Verified embedding dimension: 384

📌 Top 3 results:

--- Result 1 ---
Text: MRCET-IT 
Page 7 
 
2) Recall 
When a user decides to issue a search looking for information on a topic, the total 
database is logically divided into four segments shown in Figure 1.1. Relevant items 
 
 are those documents that contain information that helps the searcher in answering 
his question
Metadata: {'creationDate': "D:20200717150525+00'00'", 'format': 'PDF 1.5', 'source': '..\\data\\pdf\\Retrieval.pdf', 'creationdate': '2020-07-17T15:05:25+00:00', 'page': 6, 'file_path': '..\\data\\pdf\\Retrieval.pdf', 'moddate': '2020-07-17T15:05:25+00:00', 'keywords': '', 'modDate': 'D:20200717150525Z', 'creator': 'Microsoft® Word 2016', 'trapped': '', 'subject': '', 'producer': 'www.ilovepdf.com', 'total_pages': 84, 'title': '', 'author': 'mrcet'}

--- Result 2 ---
Text: MRCET-IT 
Page 81 
 
 
 
Another measure that is directly related to retrieving non-relevant

✅ Answer generated successfully

🧠 Final Answer:

Precision and recall are two major measures associated with information retrieval systems. 

- **Precision** is affected by the retrieval of non-relevant items; it drops to a number close to zero when many non-relevant items are retrieved. It measures the accuracy of the retrieved items, indicating the proportion of relevant items among all retrieved items.

- **Recall**, on the other hand, is not affected by the retrieval of non-relevant items and remains at 100 percent once achieved. It measures the ability of the system to retrieve all relevant items, indicating the proportion of relevant items that have been retrieved out of all relevant items available.

In summary, precision focuses on the relevance of retrieved items, while recall focuses on the completeness of retrieved relevant items.
